### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="mercedes_benz_greener_manufacturing",
    dataset_year="2017",
    domain_str="industry & manufacturing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/mercedes-benz-greener-manufacturing",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c mercedes-benz-greener-manufacturing -f train.csv.zip && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/mercedes_benz_greener_manufacturing && mv train.csv local-data-warehouse/mercedes_benz_greener_manufacturing/
""",
    # References
    academic_reference_bibtex=r"""@misc{Novy2017MercedesBenzGreenerManufacturing,
  author = {Alexander Novy and CH1Mercedes and Christian Drescher and Christian Pfaundler and KOESIM and Will Cukierski},
  title  = {Mercedes-Benz Greener Manufacturing},
  year   = {2017},
  howpublished = {\url{https://kaggle.com/competitions/mercedes-benz-greener-manufacturing}},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Novy2017MercedesBenzGreenerManufacturing",
    license="Kaggle Competition Rules",
    data_tags=["Non-IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized, so feature meanings are unknown.
- There is generally weird behavior related to the ID and potentially a time column (see X5 here https://www.kaggle.com/competitions/mercedes-benz-greener-manufacturing/discussion/34949). However, it is hard to parse the exact details due to anonymization. We generally take from this that the data is sorted sequentially by ID and that there may be time dependencies. Also checkout https://www.kaggle.com/code/sudalairajkumar/simple-exploration-notebook-mercedes which shows relation between index and y, showing that y goes up with higher index and it shows that X5 (the time group feature), has the highest importance. Plus, it has one outlier.
- We remove the one outlier from y.
- We drop X5 as it leaks time group information in temporal splits (besides having a full categorical-value distribution shift with a temporal split).
- We tread the ID as a time_index.
- Some of the other constructed features (X0-X8) may leak across time as well. We remove all of them for which we do not have an explanation from experts on Kaggle.
- There are exactly 4 rows that have value that is not equal to "d" in column "X4". This indicates some kind of sub-group or data different. Given the lack of semantics and anonymity, we cannot parse the meaning of "X4". Thus, to avoid this being an indicator of data leakage, we remove these rows and subsequently drop "X4" as well.
- We drop other constructed features from up to X10 for which we have no information about them and they may leak group information across rows.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="y",
    problem_type="regression",
    objective_metric_name="rmse", # r2 in competition, which is inferior to rmse for regression tasks
    time_on="time_index",
)

## Preprocessing

In [6]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

# Remove on huge y outlier that seems like a clear shift case
df = df[df["y"] < 180]

# Handle constructed features: ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']
#   - X0,X1,X2 likely only uses information per row
#   - X4 has 4 rows that are not "d", we drop these rows to avoid leakage and later drop X4
#   - X5, time group feature that leaks group time info in temporal splits
#   - X8 seems to leak group information across rows by having value assigned based on sampling from the distribution of all data, so we drop it as well
#   - We have no further information about X3 and X6, so we drop them as well to be safe, given the trend of other constructed features leaking info across rows.
#   - We drop X11 as it is a constant feature after preprocessing.
df = df[df["X4"] == "d"]

df = df.drop(columns=["X5", "X8", "X4", "X3", "X6", "X11"])
df = df.rename(columns={"ID": "time_index"})
df = df.reset_index(drop=True)

# All X features are binary/categorical
as_cat_type = [c for c in list(df) if c.startswith("X")]
df[as_cat_type] = df[as_cat_type].astype("category")

Loaded data shape: (4209, 378)


## Data Checks

In [7]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 4,204
Columns: 372

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [8]:
# Sample Rows
df_head

,time_index,y,X0,X1,X2,X10,X12,X13,X14,X15,X16,X17,X18,X19,X20,X21,X22,X23,X24,X26,X27,X28,X29,X30,X31,X32,X33,X34,X35,X36,X37,X38,X39,X40,X41,X42,X43,X44,X45,X46,X47,X48,X49,X50,X51,X52,X53,X54,X55,X56,X57,X58,X59,X60,X61,X62,X63,X64,X65,X66,X67,X68,X69,X70,X71,X73,X74,X75,X76,X77,X78,X79,X80,X81,X82,X83,X84,X85,X86,X87,X88,X89,X90,X91,X92,X93,X94,X95,X96,X97,X98,X99,X100,X101,X102,X103,X104,X105,X106,X107,X108,X109,X110,X111,X112,X113,X114,X115,X116,X117,X118,X119,X120,X122,X123,X124,X125,X126,X127,X128,X129,X130,X131,X132,X133,X134,X135,X136,X137,X138,X139,X140,X141,X142,X143,X144,X145,X146,X147,X148,X150,X151,X152,X153,X154,X155,X156,X157,X158,X159,X160,X161,X162,X163,X164,X165,X166,X167,X168,X169,X170,X171,X172,X173,X174,X175,X176,X177,X178,X179,X180,X181,X182,X183,X184,X185,X186,X187,X189,X190,X191,X192,X194,X195,X196,X197,X198,X199,X200,X201,X202,X203,X204,X205,X206,X207,X208,X209,X210,X211,X212,X213,X214,X215,X216,X217,X218,X219,X220,X221,X222,X223,X224,X225,X226,X227,X228,X229,X230,X231,X232,X233,X234,X235,X236,X237,X238,X239,X240,X241,X242,X243,X244,X245,X246,X247,X248,X249,X250,X251,X252,X253,X254,X255,X256,X257,X258,X259,X260,X261,X262,X263,X264,X265,X266,X267,X268,X269,X270,X271,X272,X273,X274,X275,X276,X277,X278,X279,X280,X281,X282,X283,X284,X285,X286,X287,X288,X289,X290,X291,X292,X293,X294,X295,X296,X297,X298,X299,X300,X301,X302,X304,X305,X306,X307,X308,X309,X310,X311,X312,X313,X314,X315,X316,X317,X318,X319,X320,X321,X322,X323,X324,X325,X326,X327,X328,X329,X330,X331,X332,X333,X334,X335,X336,X337,X338,X339,X340,X341,X342,X343,X344,X345,X346,X347,X348,X349,X350,X351,X352,X353,X354,X355,X356,X357,X358,X359,X360,X361,X362,X363,X364,X365,X366,X367,X368,X369,X370,X371,X372,X373,X374,X375,X376,X377,X378,X379,X380,X382,X383,X384,X385
0,0,130.81,k,v,at,0,0,1,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,1,1,1,0,0,0,0,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0
1,6,88.53,k,t,av,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,1,0,0,0,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0
2,7,76.26,az,w,n,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,0,1,0,1,0,0,0,0,0,1,0,0,1,0,0,0,0,1,0,0,1,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,1,1,0,1,1,1,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,0,0,1,1,0,0,0,1,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1,1,0,0,1,0,0,0,0,0,0,0,0,1,1,1,0,0,1,1,0,0,0,1,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,1,0,1,0,0,0,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,1,1,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,1,0,0,0,1,0,0,1,0,1,0,0,0,0,0,0,0,0,0

In [9]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,X0,category,0,0.0,47,"z, ak, y, ay, t, x, o, f, n, w"
1,X1,category,0,0.0,27,"aa, s, b, l, v, r, i, a, c, o"
2,X2,category,0,0.0,44,"as, ae, ai, m, ak, r, n, s, f, e"
3,X10,category,0,0.0,2,"0, 1"
4,X12,category,0,0.0,2,"0, 1"
5,X13,category,0,0.0,2,"0, 1"
6,X14,category,0,0.0,2,"0, 1"
7,X15,category,0,0.0,2,"0, 1"
8,X16,category,0,0.0,2,"0, 1"
9,X17,category,0,0.0,2,"0, 1"


In [10]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
time_index,4204.0,4208.083492,2438.254452,0.00,8417.00
y,4204.0,100.619125,12.417569,72.11,169.91


In [11]:
# Categorical Feature Statistics
cat_stats

value  count     pct
column rank                     
X0     1        z    359    8.54
       2       ak    348    8.28
       3        y    323    7.68
       4       ay    313    7.45
       5        t    306    7.28
X1     1       aa    833   19.81
       2        s    598   14.22
       3        b    592   14.08
       4        l    589   14.01
       5        v    408    9.71
X10    1        0   4148   98.67
       2        1     56    1.33
X100   1        1   2903   69.05
       2        0   1301   30.95
X101   1        1   3934   93.58
       2        0    270    6.42
X102   1        0   4175   99.31
       2        1     29    0.69
X103   1        1   3299   78.47
       2        0    905   21.53
X104   1        0   4196   99.81
       2        1      8    0.19
X105   1        0   4194   99.76
       2        1     10    0.24
X106   1        0   4149   98.69
       2        1     55    1.31
X107   1        0   4204  100.00
X108   1        0   4142   98.53
       2        1     62    1.47
X109   1        0   4034   95.96
       2        1    170    4.04
X110   1        0   4200   99.90
       2        1      4    0.10
X111   1        1   4099   97.50
       2        0    105    2.50
X112   1        0   4192   99.71
       2        1     12    0.29
X113   1        0   4111   97.79
       2        1     93    2.21
X114   1        0   3590   85.39
       2        1    614   14.61
X115   1        0   3004   71.46
       2        1   1200   28.54
X116   1        0   3377   80.33
       2        1    827   19.67
X117   1        0   3997   95.08
       2        1    207    4.92
X118   1        1   2616   62.23
       2        0   1588   37.77
X119   1        1   2616   62.23
       2        0   1588   37.77
X12    1        0   3888   92.48
       2        1    316    7.52
X120   1        1   4027   95.79
       2        0    177    4.21
X122   1        0   4174   99.29
       2        1     30    0.71
X123   1        0   4193   99.74
       2        1     11    0.26
X124   1        0   4202   99.95
       2        1      2    0.05
X125   1        0   4191   99.69
       2        1     13    0.31
X126   1        0   4041   96.12
       2        1    163    3.88
X127   1        0   2122   50.48
       2        1   2082   49.52
X128   1        1   4030   95.86
       2        0    174    4.14
X129   1        0   3689   87.75
       2        1    515   12.25
X13    1        0   3960   94.20
       2        1    244    5.80
X130   1        0   4030   95.86
       2        1    174    4.14
X131   1        0   4092   97.34
       2        1    112    2.66
X132   1        1   2895   68.86
       2        0   1309   31.14
X133   1        0   3681   87.56
       2        1    523   12.44
X134   1        0   4111   97.79
       2        1     93    2.21
X135   1        0   4090   97.29
       2        1    114    2.71
X136   1        1   4021   95.65
       2        0    183    4.35
X137   1        1   2445   58.16
       2        0   1759   41.84
X138   1        0   4032   95.91
       2        1    172    4.09
X139   1        0   3824   90.96
       2        1    380    9.04
X14    1        0   2403   57.16
       2        1   1801   42.84
X140   1        0   4034   95.96
       2        1    170    4.04
X141   1        0   4144   98.57
       2        1     60    1.43
X142   1        1   3239   77.05
       2        0    965   22.95
X143   1        0   4043   96.17
       2        1    161    3.83
X144   1        1   3398   80.83
       2        0    806   19.17
X145   1        0   4198   99.86
       2        1      6    0.14
X146   1        0   4032   95.91
       2        1    172    4.09
X147   1        0   4111   97.79
       2        1     93    2.21
X148   1        0   4015   95.50
       2        1    189    4.50
X15    1        0   4202   99.95
       2        1      2    0.05
X150   1        1   3331   79.23
       2        0    873   20.77
X151   1        0   3844   91.44
       2        1    360    8.56
X152   1        0  

In [12]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.738,0.278,154.196,0.015,log,57845.6,60798.8,exponential


## Task Curation

In [13]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from sklearn.model_selection import TimeSeriesSplit

df = df.sort_values(by="time_index", ascending=True).reset_index(drop=True)

default_train_index = df.index[:1000].to_list()
to_split_index = df.index[1000:]

spliter = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=9, test_size=None)
splits = {}
for i, (train_idx, test_idx) in enumerate(spliter.split(to_split_index)):
    print(len(train_idx) + len(default_train_index), len(test_idx))
    splits[i] = {0: (default_train_index + to_split_index[train_idx].to_list(), to_split_index[test_idx].tolist())}

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="This is a small temporal dataset. Thus, we could refit often and we could simulate that a model was often refit, basically, if possible, after each time_index (akin to LOO). As this would create too many splits with too small test sizes for a robust signal, we opt for less splits. Moreover, as we want to have sufficient train data signal, we start with the first 1000 samples of the data as fixed train data. The rest is than gradually split to create 9 test splits using a sklearn TimeSeriesSplit.",
    splits=splits,
)

1324 320
1644 320
1964 320
2284 320
2604 320
2924 320
3244 320
3564 320
3884 320


## Export

In [14]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c0e8e-8749-7ff7-9c06-632c3ca2aa05
3898208b0b8c66d56b9a9b3da81335fc1d0a60f342f4a6ecb51898fa086c7cbf
